In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))

response = llm.invoke("Explain what LangChain is in 2 sentences")
print(response.content)

LangChain is a framework designed to facilitate the development of applications that use language models, enabling developers to easily integrate and manage these models in various workflows. It provides tools for managing chains of language model calls, handling memory, and connecting to data sources, making it easier to build complex applications powered by natural language processing.


In [4]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    "Explain Image Processing in funny style, in 2 sentences."
)

chain = prompt|llm

response = chain.invoke({"topic": "vector databases", "style": "simple"})
print(response.content)

Image processing is like putting a pair of funky sunglasses on your pictures—suddenly, they look cooler, sharper, and ready for the catwalk! It's the digital equivalent of a makeover show, turning "meh" visuals into "wow" masterpieces, one pixel at a time!


In [5]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

embeddings = OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY"))

documents = [
    "Tech companies laid off thousands of employees in 2023",
    "Many workers lost their jobs in the technology sector",
    "I love eating pizza on weekends",
    "AI is transforming how businesses automate decisions",
    "Oracle announced major layoffs affecting India operations",
    "The stock market crashed due to inflation concerns"
]

vectorstore = Chroma.from_texts(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("companies firing workers")
for doc in results:
    print(doc.page_content)

/var/folders/5w/g3jsjkns29b_0gg00vxczghr0000gn/T/ipykernel_97285/1713605881.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Many workers lost their jobs in the technology sector
Tech companies laid off thousands of employees in 2023
Oracle announced major layoffs affecting India operations


In [6]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Answer the question using only this context:\n{context}\n\nQuestion: {question}"
)

def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Which companies had layoffs?")
print(answer)

Tech companies, including Oracle, had layoffs in 2023.
